# Hybrid RAG Application

In this notebook, we'll implement and test a complete Retrieval-Augmented Generation (RAG) system that uses a hybrid retrieval approach.


## Table of Contents:

- **Part 1: Setup and Data Preparation**
  - Imports and Utilities
  - Loading and Chunking Documents
  - OpenAI API Key Setup
- **Part 2: Initializing the Retrievers**
  - The Dense Retriever (VectorDB)
  - The Sparse Retriever (BM25)
  - The Hybrid Retriever (RRF)
- **Part 3: The Generic RAG Pipeline**
  - Refactoring the Pipeline
- **Part 4: Testing and Comparison**
  - Test 1: Dense Retrieval (Semantic Search)
  - Test 2: Sparse Retrieval (Keyword Search)
  - Test 3: Hybrid Retrieval


## Part 1: Setup and Data Preparation

### Imports and Utilities
We're importing the necessary libraries


In [1]:
import os
import openai
import asyncio
import nest_asyncio
from getpass import getpass

from aimakerspace.vectordatabase import VectorDatabase
from aimakerspace.openai_utils.chatmodel import ChatOpenAI
from aimakerspace.retrievers import BM25Retriever, HybridRetriever
from aimakerspace.text_utils import TextFileLoader, CharacterTextSplitter
from aimakerspace.openai_utils.prompts import SystemRolePrompt, UserRolePrompt

nest_asyncio.apply()

### Loading and Chunking Documents
First, we load our source file (`PMarcaBlogs.txt`) and split it into smaller "document" chunks. This is a crucial step for any RAG system.

In [2]:
text_loader = TextFileLoader("data/PMarcaBlogs.txt")
documents = text_loader.load_documents()
text_splitter = CharacterTextSplitter()
split_documents = text_splitter.split_texts(documents)
print(f"Loaded and split the source file into {len(split_documents)} documents.")

Loaded and split the source file into 373 documents.


### OpenAI API Key Setup
We need to provide our OpenAI API Key to create embeddings for our dense retriever.

In [3]:
openai.api_key = getpass("OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = openai.api_key

## Part 2: Initializing the Retrievers

Now, we will initialize our three different retrievers. Each will be ready to process queries, but they will find documents in very different ways.

In [4]:
dense_retriever = VectorDatabase()
dense_retriever = asyncio.run(dense_retriever.abuild_from_list(split_documents))
print("Dense Retriever is ready.")

sparse_retriever = BM25Retriever(corpus=split_documents)
print("Sparse Retriever is ready.")

hybrid_retriever = HybridRetriever(dense_retriever, sparse_retriever)
print("Hybrid Retriever is ready.")

Dense Retriever is ready.
Sparse Retriever is ready.
Hybrid Retriever is ready.


## Part 3: The Generic RAG Pipeline

Pipeline accepts any retriever object. This makes our pipeline flexible and modular.

In [5]:
class RetrievalAugmentedQAPipeline:
    def __init__(self, llm: ChatOpenAI, retriever) -> None:
        self.llm = llm
        self.retriever = retriever

    def run_pipeline(self, user_query: str, k: int = 3) -> dict:
        context_list = self.retriever.search_by_text(user_query, k=k)
        
        context_prompt = ""
        for i, (context, score) in enumerate(context_list, 1):
            context_prompt += f"[Source {i}, Score: {score:.4f}]: {context}\n\n"
        
        system_prompt_template = "You are a knowledgeable assistant. Answer questions based only on the provided context. If the context doesn't contain the answer, say 'I don't know'."
        user_prompt_template = "Context Information:\n{context}\n\nQuestion: {user_query}"
        
        system_prompt = SystemRolePrompt(system_prompt_template)
        user_prompt = UserRolePrompt(user_prompt_template)

        response = self.llm.run([
            system_prompt.create_message(), 
            user_prompt.create_message(context=context_prompt.strip(), user_query=user_query)
        ])
        
        return {
            "response": response, 
            "context": context_list
        }

chat_openai = ChatOpenAI()

## Part 4: Testing and Comparison

Let's run a few tests to see how each retriever performs

**Our Queries:**
1.  **Keyword Query:** `"What is the Michael Eisner Memorial Weak Executive Problem?"` (Tests precision on specific terms).
2.  **Conceptual Query:** `"How can a company prevent its top talent from leaving?"` (Tests understanding of meaning).



In [6]:
query_keyword = "What is the Michael Eisner Memorial Weak Executive Problem?"
query_conceptual = "How can a company prevent its top talent from leaving?"

def print_results(title: str, results: dict):
    print(f"--- {title} ---")
    print(f"LLM Response: {results['response']}")
    print("\n--- Context Used ---")
    for doc, score in results["context"]:
        print(f"Score: {score:.4f}\n{doc[:120]}...\n")
    print("-" * 20)

# --- Initialize Pipelines ---
rag_pipeline_dense = RetrievalAugmentedQAPipeline(llm=chat_openai, retriever=dense_retriever)
rag_pipeline_sparse = RetrievalAugmentedQAPipeline(llm=chat_openai, retriever=sparse_retriever)
rag_pipeline_hybrid = RetrievalAugmentedQAPipeline(llm=chat_openai, retriever=hybrid_retriever)

### Comparison 1: The Keyword Query

Here, we expect the sparse and hybrid retrievers to perform best, as this query contains very specific keywords.

In [7]:
print("--- RUNNING KEYWORD QUERY COMPARISON ---")

# Run with Dense
results_dense = rag_pipeline_dense.run_pipeline(query_keyword)
print_results("Dense Retriever (Keyword Query)", results_dense)

# Run with Sparse
results_sparse = rag_pipeline_sparse.run_pipeline(query_keyword)
print_results("Sparse Retriever (Keyword Query)", results_sparse)

# Run with Hybrid
results_hybrid = rag_pipeline_hybrid.run_pipeline(query_keyword)
print_results("Hybrid Retriever (Keyword Query)", results_hybrid)

--- RUNNING KEYWORD QUERY COMPARISON ---
--- Dense Retriever (Keyword Query) ---
LLM Response: The Michael Eisner Memorial Weak Executive Problem refers to the tendency of CEOs or startup founders to hire weak executives in areas where they themselves have expertise. This phenomenon occurs because they may want to maintain control or remain "the man" in that function instead of bringing in a strong leader, thereby leading to a situation where the executive hired is inadequate for the role. An example cited is Michael Eisner, who was a successful TV network executive but struggled when he bought ABC at Disney, resulting in poor performance.

--- Context Used ---
Score: 0.6539
ordingly.
Seventh, when hiring the executive to run your former specialty, be
careful you don’t hire someone weak on pur...

Score: 0.5036
m. They have areas where they are truly deXcient in judgment or skill set. That’s just life. Almost nobody is brilliant
...

Score: 0.4814
ed?
In reality — as opposed to Marc’s 

### Comparison 2: The Conceptual Query

Now, we expect the dense and hybrid retrievers to excel, as this query requires understanding the *meaning* of "top talent leaving" without exact keywords.

In [8]:
print("--- RUNNING CONCEPTUAL QUERY COMPARISON ---")

# Run with Dense
results_dense = rag_pipeline_dense.run_pipeline(query_conceptual)
print_results("Dense Retriever (Conceptual Query)", results_dense)

# Run with Sparse
results_sparse = rag_pipeline_sparse.run_pipeline(query_conceptual)
print_results("Sparse Retriever (Conceptual Query)", results_sparse)

# Run with Hybrid
results_hybrid = rag_pipeline_hybrid.run_pipeline(query_conceptual)
print_results("Hybrid Retriever (Conceptual Query)", results_hybrid)

--- RUNNING CONCEPTUAL QUERY COMPARISON ---
--- Dense Retriever (Conceptual Query) ---
LLM Response: A company can prevent its top talent from leaving by promoting its best people into higher positions, simplifying and clarifying its organizational structure, and addressing any mediocrity by letting go of underperformers. It's also important to create a winning environment where employees want to stay, as companies that are winning typically do not experience retention problems.

--- Context Used ---
Score: 0.5235
s to a
challenge. Again, a classic problem for the former hot
startup. Look particularly hard at the people who joined i...

Score: 0.5205
 dancing
in her head.
You can oaen defeat this by simply explaining the realities of
the compensation package she’s bein...

Score: 0.5167
ining great people, particularly at big companies in industries like technology, where stock options matter
and where pe...

--------------------
--- Sparse Retriever (Conceptual Query) ---
LLM Response

## Part 5: Analyzing the Hybrid Retriever's Brain

The final and most important test is to look "under the hood" of the hybrid retriever to see how it's making its decisions. We'll use our `detailed_results=True` flag to see where the top documents came from.

In [ ]:
# --- Analyze the Keyword Query ---
detailed_results_keyword = hybrid_retriever.search_by_text(query_keyword, detailed_results=True)

print("--- Hybrid Metrics (Keyword Query: Eisner Problem) ---")
for result in detailed_results_keyword:
    print(f"Score: {result['score']:.4f}")
    print(f"Sources: {result['sources']}")
    print(f"Document: {result['doc'][:150]}...\n")

# --- Analyze the Conceptual Query ---
detailed_results_conceptual = hybrid_retriever.search_by_text(query_conceptual, detailed_results=True)

print("\n--- Hybrid Metrics (Conceptual Query: Talent Retention) ---")
for result in detailed_results_conceptual:
    print(f"Score: {result['score']:.4f}")
    print(f"Sources: {result['sources']}")
    print(f"Document: {result['doc'][:150]}...\n")

--- Hybrid Metrics (Keyword Query: Eisner Problem) ---
Score: 0.0328
Sources: ['sparse', 'dense']
Document: ordingly.
Seventh, when hiring the executive to run your former specialty, be
careful you don’t hire someone weak on purpose.
This sounds silly, but y...

Score: 0.0161
Sources: ['dense']
Document: m. They have areas where they are truly deXcient in judgment or skill set. That’s just life. Almost nobody is brilliant
at everything. When hiring and...

Score: 0.0161
Sources: ['sparse']
Document: en has a hard time letting
go of the function that brought him to the party. The result: you
hire someone weak into the executive role for that functi...


--- Hybrid Metrics (Conceptual Query: Talent Retention) ---
Score: 0.0164
Sources: ['dense']
Document: s to a
challenge. Again, a classic problem for the former hot
startup. Look particularly hard at the people who joined in
the two years following the ...

Score: 0.0164
Sources: ['sparse']
Document: ions combine in a whirlwind
of violen